In [1]:
from pathlib import Path

import anndata as ad

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

qc_data_path = (
    project_root / "data" / "interim" / "GSE176160_patients_qc.h5ad"
)

adata = ad.read_h5ad(qc_data_path, backed="r")

print("QC dataset shape:", adata.shape)
print("Donors:", adata.obs["Donor"].nunique())
print("Sensitivity cells:", adata.obs["keep_sensitivity"].sum())

QC dataset shape: (30589, 58735)
Donors: 12
Sensitivity cells: 30576


In [2]:
import numpy as np
import pandas as pd

raw_totals = np.zeros(adata.n_obs, dtype=np.float64)

for counts_chunk, start, stop in adata.chunked_X(chunk_size=256):
    chunk_totals = counts_chunk.sum(axis=1, dtype=np.float64)
    raw_totals[start:stop] = np.asarray(chunk_totals).ravel()

assert (raw_totals > 0).all(), "A cell has a nonpositive count total."

qc_totals = adata.obs["total_counts"].to_numpy()

print("Totals match QC metadata:", np.array_equal(raw_totals, qc_totals))

pd.Series(raw_totals, name="raw_total_counts").describe().round(2)

Totals match QC metadata: True


count    30589.00
mean     13214.98
std       5839.06
min       2861.00
25%       9356.00
50%      12369.00
75%      16146.00
max      70048.00
Name: raw_total_counts, dtype: float64

In [3]:
assert np.array_equal(raw_totals, qc_totals), (
    "Count totals do not match the QC metadata."
)

target_sum = 10_000
normalization_factors = target_sum / raw_totals

normalization_preview = pd.DataFrame(
    {
        "raw_total_counts": raw_totals,
        "normalization_factor": normalization_factors,
    },
    index=adata.obs_names,
)

normalization_preview.head().round(4)

,raw_total_counts,normalization_factor
index,,
AAACCTGCACGTCTCT,17378.0,0.5754
AAACCTGCAGTATCTG,13234.0,0.7556
AAACCTGTCCAGTATG,19998.0,0.5001
AAACCTGTCCCAACGG,15525.0,0.6441
AAACCTGTCCCGGATG,17370.0,0.5757


In [4]:
n_preview_cells = 5

preview_counts = adata.X[:n_preview_cells, :]
preview_factors = normalization_factors[:n_preview_cells]

normalized_preview = preview_counts.multiply(
    preview_factors[:, None]
).tocsr()

normalized_totals = np.asarray(
    normalized_preview.sum(axis=1)
).ravel()

assert np.allclose(normalized_totals, target_sum), (
    "Normalized totals differ from the target."
)

normalization_preview.iloc[:n_preview_cells].assign(
    normalized_total_counts=normalized_totals
).round(4)

,raw_total_counts,normalization_factor,normalized_total_counts
index,,,
AAACCTGCACGTCTCT,17378.0,0.5754,10000.0
AAACCTGCAGTATCTG,13234.0,0.7556,10000.0
AAACCTGTCCAGTATG,19998.0,0.5001,10000.0
AAACCTGTCCCAACGG,15525.0,0.6441,10000.0
AAACCTGTCCCGGATG,17370.0,0.5757,10000.0


In [5]:
from scipy import sparse

normalized_chunks = []

for counts_chunk, start, stop in adata.chunked_X(chunk_size=256):
    chunk_factors = normalization_factors[start:stop]

    normalized_chunk = (
        counts_chunk
        .multiply(chunk_factors[:, None])
        .astype(np.float32)
        .tocsr()
    )

    normalized_chunks.append(normalized_chunk)

normalized_counts = sparse.vstack(normalized_chunks, format="csr")
del normalized_chunks

normalized_total_counts = np.asarray(
    normalized_counts.sum(axis=1, dtype=np.float64)
).ravel()

assert normalized_counts.shape == adata.shape
assert np.allclose(
    normalized_total_counts, target_sum, rtol=1e-6, atol=0
), "Some normalized totals differ from the target."

print("Normalized matrix shape:", normalized_counts.shape)
print("Matrix dtype:", normalized_counts.dtype)
print(
    "Largest deviation from target:",
    np.abs(normalized_total_counts - target_sum).max(),
)

Normalized matrix shape: (30589, 58735)
Matrix dtype: float32
Largest deviation from target: 0.00036454200744628906


In [6]:
log_normalized_counts = normalized_counts.copy()

np.log1p(
    log_normalized_counts.data,
    out=log_normalized_counts.data,
)

assert np.isfinite(log_normalized_counts.data).all(), (
    "The log-transformed matrix contains non-finite values."
)

print("Log-normalized matrix shape:", log_normalized_counts.shape)

pd.DataFrame(
    {
        "normalized_count": normalized_counts.data[:5],
        "log1p_value": log_normalized_counts.data[:5],
    }
).round(4)

Log-normalized matrix shape: (30589, 58735)


,normalized_count,log1p_value
0,8.6316,2.2650
1,0.5754,0.4545
2,0.5754,0.4545
3,1.1509,0.7659
4,3.4526,1.4935


In [7]:
normalized_adata = ad.AnnData(
    X=log_normalized_counts,
    obs=adata.obs.copy(),
    var=adata.var.copy(),
)

normalized_adata.obs["normalization_factor"] = normalization_factors

normalized_adata.uns["normalization"] = {
    "method": "total_count_scaling_then_log1p",
    "target_sum": target_sum,
    "counts_source": qc_data_path.name,
}

assert normalized_adata.obs_names.equals(adata.obs_names)
assert normalized_adata.var_names.equals(adata.var_names)

print("Normalized AnnData shape:", normalized_adata.shape)
print("Normalization settings:", normalized_adata.uns["normalization"])

Normalized AnnData shape: (30589, 58735)
Normalization settings: {'method': 'total_count_scaling_then_log1p', 'target_sum': 10000, 'counts_source': 'GSE176160_patients_qc.h5ad'}


In [8]:
lane_normalization_metrics = normalized_adata.obs[
    ["Lane", "normalization_factor"]
].assign(
    raw_total_counts=raw_totals,
    normalized_total_counts=normalized_total_counts,
)

lane_normalization_summary = (
    lane_normalization_metrics
    .groupby("Lane", observed=True)
    .agg(
        n_cells=("raw_total_counts", "size"),
        median_raw_total=("raw_total_counts", "median"),
        median_normalized_total=("normalized_total_counts", "median"),
        median_factor=("normalization_factor", "median"),
    )
)

lane_normalization_summary.round(4)

,n_cells,median_raw_total,median_normalized_total,median_factor
Lane,,,,
1,4816,13583.5,10000.0,0.7362
2,4924,13960.0,10000.0,0.7163
3,5123,13444.0,10000.0,0.7438
4,5161,11192.0,10000.0,0.8935
5,5279,11591.0,10000.0,0.8627
6,5286,11001.5,10000.0,0.9090


In [9]:
normalized_path = (
    project_root
    / "data"
    / "interim"
    / "GSE176160_patients_normalized.h5ad"
)
temporary_path = normalized_path.with_suffix(".tmp.h5ad")

normalized_adata.write_h5ad(temporary_path, compression="gzip")

saved_adata = ad.read_h5ad(temporary_path, backed="r")

try:
    assert saved_adata.shape == normalized_adata.shape
    assert saved_adata.obs_names.equals(normalized_adata.obs_names)
    assert saved_adata.var_names.equals(normalized_adata.var_names)
    assert saved_adata.uns["normalization"]["target_sum"] == target_sum
finally:
    saved_adata.file.close()

temporary_path.replace(normalized_path)

lane_summary_path = (
    project_root / "results" / "tables" / "normalization_by_lane.csv"
)
lane_normalization_summary.to_csv(lane_summary_path)

print("Verified shape:", normalized_adata.shape)
print("Saved normalized dataset:", normalized_path.name)
print("Saved lane summary:", lane_summary_path.name)

Verified shape: (30589, 58735)
Saved normalized dataset: GSE176160_patients_normalized.h5ad
Saved lane summary: normalization_by_lane.csv


## Normalization outcome

The quality-controlled dataset contains 30,589 cells and 58,735 genes from 12 donors. Counts were scaled to a target sum of 10,000 per cell, then transformed with `log1p`. The largest deviation from the target before log transformation was approximately 0.000365 counts.

Median raw count totals differed across lanes. After scaling, the median total in each of the six lanes was approximately 10,000. PCA will be used to assess whether gene expression patterns still vary by lane.

The normalized expression matrix was saved to `data/interim/GSE176160_patients_normalized.h5ad`. The lane summary was saved to `results/tables/normalization_by_lane.csv`. The original counts for count-based statistical analyses are available in `data/interim/GSE176160_patients_qc.h5ad`.